# Pipeline de Limpeza — ClickGalpões

Pipeline: raw → clean via stages compartilhadas + especificas ClickGalpoes.

In [ ]:
import datawork
datawork.setup()

In [ ]:
from datawork.io.loaders import load_scraping_results

FONTE_ID = ""  # Preencher com UUID
df_raw = load_scraping_results(FONTE_ID)
print(f"{len(df_raw)} registros raw")

In [ ]:
from datawork.pipeline import PipelineRunner
from datawork.pipeline.stages import (
    drop_empty_rows, normalize_areas, geocode_addresses,
    compute_dedup_hash, drop_duplicates_by_hash,
    generate_titulo, rename_to_silver, select_columns,
)
from datawork.pipeline.stages_scraping import (
    normalize_scraped_titles, normalize_price_scraped,
    detect_tipo_from_title, extract_features_from_description,
    fill_source_metadata,
)
from datawork.pipeline.stages_clickgalpoes import (
    parse_nextjs_hydration, extract_condominium_name,
    normalize_imageboss_urls,
)

pipeline = (
    PipelineRunner("clickgalpoes")
    .add("hydration", parse_nextjs_hydration)
    .add("cond_name", extract_condominium_name)
    .add("empty", drop_empty_rows)
    .add("titles", normalize_scraped_titles)
    .add("tipo", detect_tipo_from_title)
    .add("areas", normalize_areas)
    .add("prices", normalize_price_scraped)
    .add("features", extract_features_from_description)
    .add("images", normalize_imageboss_urls)
    .add("geocode", geocode_addresses)
    .add("metadata", lambda df: fill_source_metadata(df, "ClickGalpoes", "clickgalpoes.com.br"))
    .add("titulo", generate_titulo)
    .add("dedup", compute_dedup_hash)
    .add("dedup_drop", drop_duplicates_by_hash)
    .add("rename", rename_to_silver)
    .add("select", select_columns)
)

df_clean = pipeline.run(df_raw)
pipeline.summary()

In [ ]:
from datawork.display import show_sample, show_stats
show_sample(df_clean, n=5)

In [ ]:
# Validar contra Silver schema
from datawork.contracts.silver import CleanRecordSchema
CleanRecordSchema.validate(df_clean)
print("Validacao Silver OK")

In [ ]:
# Push para API
# from datawork.io.pushers import push_clean_to_api
# push_clean_to_api(df_clean, FONTE_ID)